In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file()
)

RAW_DIR = ROOT / "data/raw/viadukten"
PROCESSED_DIR = ROOT / "data/processed"

print("Project root:", ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

Project root: /Users/shabboo/Documents/BIM Dashboard/silverlinjen-bim-analytics
Raw data: /Users/shabboo/Documents/BIM Dashboard/silverlinjen-bim-analytics/data/raw/viadukten
Processed data: /Users/shabboo/Documents/BIM Dashboard/silverlinjen-bim-analytics/data/processed


In [2]:
csv_files = sorted(RAW_DIR.glob("*.csv"))

for file in csv_files:
    print(f"{file.name} | {file.stat().st_size} bytes")

print("\nTotal CSV files:", len(csv_files))

QTO - Floors.csv | 265 bytes
QTO - Structural Foundations.csv | 1680 bytes
QTO - Structural Framing.csv | 854 bytes
QTO - Walls.csv | 505 bytes
structural_columns.csv | 1689 bytes

Total CSV files: 5


In [3]:
csv_datasets = {}

for file in csv_files:
    df = pd.read_csv(
        file,
        encoding="utf-8-sig"
    )

    csv_datasets[file.stem] = df

    print(f"\n--- {file.name} ---")
    print("Shape:", df.shape)

    display(df.head(3))


--- QTO - Floors.csv ---
Shape: (1, 10)


,Mark,Family and Type,Structural Material,Level,Height_Offset_m,Core_Thickness_m,Default_Thickness_m,Area_m2,Volume_m3,Count
0,NaN,Floor: STI_Bro 1500,"Concrete, Cast-in-Place gray",Brobana,0.0,1.5,1.5,2430.0,3645.0,1



--- QTO - Structural Foundations.csv ---
Shape: (12, 10)


,Mark,Family and Type,Structural Material,Level,Length_m,Width_m,Thickness_m,Height_Offset_m,Volume_m3,Count
0,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,4.0,4.0,1.5,0.0,24.0,1
1,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,4.0,4.0,1.5,0.0,24.0,1
2,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,4.0,4.0,1.5,0.0,24.0,1



--- QTO - Structural Framing.csv ---
Shape: (6, 8)


,Mark,Family and Type,Structural Material,Reference Level,System_Length_m,Cut_Length_m,Volume_m3,Count
0,NaN,Concrete-Rectangular Beam: STI_Balk 1600x700,"Concrete, Cast-in-Place gray",Mark 0,12.4,12.4,13.888,1
1,NaN,Concrete-Rectangular Beam: STI_Balk 1600x700,"Concrete, Cast-in-Place gray",Mark 0,12.4,12.4,13.888,1
2,NaN,Concrete-Rectangular Beam: STI_Balk 1600x700,"Concrete, Cast-in-Place gray",Mark 0,12.4,12.4,13.888,1



--- QTO - Walls.csv ---
Shape: (2, 13)


,Mark,Family and Type,Structural Material,Base Constraint,Top Constraint,Base_Offset_m,Top_Offset_m,Length_m,Width_m,Height_m,Area_m2,Volume_m3,Count
0,NaN,Basic Wall: STI_Betong 1500,"Concrete, Cast-in-Place gray",Mark 0,Up to level: Brobana,-1.5,0.0,16.2,1.5,9.1,147.42,221.13,1
1,NaN,Basic Wall: STI_Betong 1500,"Concrete, Cast-in-Place gray",Mark 0,Up to level: Brobana,-1.5,0.0,16.2,1.5,9.1,147.42,221.13,1



--- structural_columns.csv ---
Shape: (12, 9)


,Mark,Family and Type,Structural Material,Base Level,Top Level,System_Length_m,Cut_Length_m,Volume_m3,Count
0,NaN,M_Concrete-Rectangular-Column: STI_Betong 1200...,"Concrete, Cast-in-Place gray",Mark 0,Mark 0,5.2,NaN,7.488,1
1,NaN,M_Concrete-Rectangular-Column: STI_Betong 1200...,"Concrete, Cast-in-Place gray",Mark 0,Mark 0,5.2,NaN,7.488,1
2,NaN,M_Concrete-Rectangular-Column: STI_Betong 1200...,"Concrete, Cast-in-Place gray",Mark 0,Mark 0,5.2,NaN,7.488,1


In [4]:
category_mapping = {
    "QTO - Floors": "Bridge Deck",
    "QTO - Structural Foundations": "Foundations",
    "QTO - Structural Framing": "Beams",
    "QTO - Walls": "Walls",
    "structural_columns": "Columns",
}

for name in csv_datasets:
    category = category_mapping.get(name, "Unclassified")
    print(f"{name} -> {category}")

QTO - Floors -> Bridge Deck
QTO - Structural Foundations -> Foundations
QTO - Structural Framing -> Beams
QTO - Walls -> Walls
structural_columns -> Columns


In [5]:
def normalize_column_name(name):
    return (
        name.strip()
        .lower()
        .replace(" ", "_")
    )


normalized_datasets = {}

for name, df in csv_datasets.items():
    normalized = df.copy()

    normalized.columns = [
        normalize_column_name(col)
        for col in normalized.columns
    ]

    normalized["category"] = category_mapping.get(
        name, "Unclassified"
    )

    normalized["source_file"] = f"{name}.csv"

    normalized_datasets[name] = normalized

    print(f"\n--- {name} ---")
    print(normalized.columns.tolist())


--- QTO - Floors ---
['mark', 'family_and_type', 'structural_material', 'level', 'height_offset_m', 'core_thickness_m', 'default_thickness_m', 'area_m2', 'volume_m3', 'count', 'category', 'source_file']

--- QTO - Structural Foundations ---
['mark', 'family_and_type', 'structural_material', 'level', 'length_m', 'width_m', 'thickness_m', 'height_offset_m', 'volume_m3', 'count', 'category', 'source_file']

--- QTO - Structural Framing ---
['mark', 'family_and_type', 'structural_material', 'reference_level', 'system_length_m', 'cut_length_m', 'volume_m3', 'count', 'category', 'source_file']

--- QTO - Walls ---
['mark', 'family_and_type', 'structural_material', 'base_constraint', 'top_constraint', 'base_offset_m', 'top_offset_m', 'length_m', 'width_m', 'height_m', 'area_m2', 'volume_m3', 'count', 'category', 'source_file']

--- structural_columns ---
['mark', 'family_and_type', 'structural_material', 'base_level', 'top_level', 'system_length_m', 'cut_length_m', 'volume_m3', 'count', 'cat

In [6]:
required_columns = {
    "family_and_type",
    "structural_material",
    "volume_m3",
    "count",
    "category",
    "source_file",
}

for name, df in normalized_datasets.items():
    missing = required_columns - set(df.columns)

    if missing:
        print(f"FAILED: {name} | Missing: {missing}")
    else:
        print(f"PASSED: {name}")

PASSED: QTO - Floors
PASSED: QTO - Structural Foundations
PASSED: QTO - Structural Framing
PASSED: QTO - Walls
PASSED: structural_columns


In [7]:
df_revit = pd.concat(
    normalized_datasets.values(),
    ignore_index=True,
    sort=False
)

display(df_revit)

print("Dataset shape:", df_revit.shape)
print("Total rows:", len(df_revit))

,mark,family_and_type,structural_material,level,height_offset_m,core_thickness_m,default_thickness_m,area_m2,volume_m3,count,...,reference_level,system_length_m,cut_length_m,base_constraint,top_constraint,base_offset_m,top_offset_m,height_m,base_level,top_level
0,NaN,Floor: STI_Bro 1500,"Concrete, Cast-in-Place gray",Brobana,0.0,1.5,1.5,2430.00,3645.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,0.0,NaN,NaN,NaN,24.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,0.0,NaN,NaN,NaN,24.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,0.0,NaN,NaN,NaN,24.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,0.0,NaN,NaN,NaN,24.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,0.0,NaN,NaN,NaN,24.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,0.0,NaN,NaN,NaN,24.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,0.0,NaN,NaN,NaN,24.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,0.0,NaN,NaN,NaN,24.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",Mark 0,0.0,NaN,NaN,NaN,24.000,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Dataset shape: (33, 25)
Total rows: 33


In [8]:
df_revit = df_revit.rename(columns={
    "family_and_type": "name",
    "structural_material": "material",
})

numeric_columns = [
    col for col in df_revit.columns
    if col.endswith(("_m", "_m2", "_m3"))
]

numeric_columns.append("count")

for col in numeric_columns:
    df_revit[col] = pd.to_numeric(
        df_revit[col],
        errors="raise"
    )

df_revit["source"] = "Revit CSV"

display(
    df_revit[
        [
            "category",
            "name",
            "material",
            "volume_m3",
            "count",
            "source_file"
        ]
    ]
)

,category,name,material,volume_m3,count,source_file
0,Bridge Deck,Floor: STI_Bro 1500,"Concrete, Cast-in-Place gray",3645.000,1,QTO - Floors.csv
1,Foundations,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",24.000,1,QTO - Structural Foundations.csv
2,Foundations,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",24.000,1,QTO - Structural Foundations.csv
3,Foundations,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",24.000,1,QTO - Structural Foundations.csv
4,Foundations,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",24.000,1,QTO - Structural Foundations.csv
5,Foundations,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",24.000,1,QTO - Structural Foundations.csv
6,Foundations,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",24.000,1,QTO - Structural Foundations.csv
7,Foundations,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",24.000,1,QTO - Structural Foundations.csv
8,Foundations,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",24.000,1,QTO - Structural Foundations.csv
9,Foundations,M_Footing-Rectangular: STI_Plint 4000,"Concrete, Cast-in-Place gray",24.000,1,QTO - Structural Foundations.csv


In [9]:
csv_quality_summary = {
    "total_rows": len(df_revit),

    "total_elements": df_revit["count"].sum(),

    "missing_materials": df_revit[
        "material"
    ].isna().sum(),

    "missing_volumes": df_revit[
        "volume_m3"
    ].isna().sum(),

    "non_finite_volumes": (
        ~np.isfinite(df_revit["volume_m3"])
    ).sum(),

    "non_positive_volumes": (
        df_revit["volume_m3"] <= 0
    ).sum(),

    "invalid_counts": (
        df_revit["count"] != 1
    ).sum(),

    "missing_marks": (
        df_revit["mark"].isna()
        | df_revit["mark"].astype(str).str.strip().eq("")
    ).sum(),

    "unclassified_elements": (
        df_revit["category"] == "Unclassified"
    ).sum(),
}

display(
    pd.DataFrame(
        csv_quality_summary.items(),
        columns=["metric", "value"]
    )
)


,metric,value
0,total_rows,33
1,total_elements,33
2,missing_materials,0
3,missing_volumes,0
4,non_finite_volumes,0
5,non_positive_volumes,0
6,invalid_counts,0
7,missing_marks,33
8,unclassified_elements,0


In [10]:
assert df_revit["count"].eq(1).all()
assert np.isfinite(df_revit["volume_m3"]).all()
assert df_revit["volume_m3"].gt(0).all()

df_revit_qto = (
    df_revit
    .groupby("category")
    .agg(
        element_count=("count", "sum"),
        total_volume_m3=("volume_m3", "sum"),
        missing_volumes=(
            "volume_m3",
            lambda values: values.isna().sum()
        )
    )
    .reset_index()
    .sort_values(
        "total_volume_m3",
        ascending=False
    )
)

display(df_revit_qto.round(3))

print(
    "Total Revit volume:",
    round(df_revit_qto["total_volume_m3"].sum(), 3),
    "m³"
)

,category,element_count,total_volume_m3,missing_volumes
1,Bridge Deck,1,3645.000,0
4,Walls,2,442.260,0
3,Foundations,12,288.000,0
2,Columns,12,89.856,0
0,Beams,6,83.328,0


Total Revit volume: 4548.444 m³


In [11]:
df_ifc_qto = pd.read_csv(
    PROCESSED_DIR / "ifc_quantity_summary.csv"
)

display(df_ifc_qto)

,category,element_count,total_volume_m3,missing_volumes
0,Bridge Deck,1,3645.000,0
1,Walls,2,442.260,0
2,Foundations,12,288.000,0
3,Columns,12,89.856,0
4,Beams,6,83.328,0


In [12]:
df_comparison = df_ifc_qto.merge(
    df_revit_qto,
    on="category",
    how="outer",
    suffixes=("_ifc", "_revit"),
    validate="one_to_one",
    indicator=True
)

df_comparison["count_difference"] = (
    df_comparison["element_count_ifc"]
    - df_comparison["element_count_revit"]
)

df_comparison["volume_difference_m3"] = (
    df_comparison["total_volume_m3_ifc"]
    - df_comparison["total_volume_m3_revit"]
)

display(df_comparison.round(6))

,category,element_count_ifc,total_volume_m3_ifc,missing_volumes_ifc,element_count_revit,total_volume_m3_revit,missing_volumes_revit,_merge,count_difference,volume_difference_m3
0,Beams,6,83.328,0,6,83.328,0,both,0,0.0
1,Bridge Deck,1,3645.000,0,1,3645.000,0,both,0,0.0
2,Columns,12,89.856,0,12,89.856,0,both,0,0.0
3,Foundations,12,288.000,0,12,288.000,0,both,0,-0.0
4,Walls,2,442.260,0,2,442.260,0,both,0,-0.0


In [13]:
VOLUME_TOLERANCE = 0.001

df_comparison["count_match"] = (
    df_comparison["count_difference"] == 0
)

df_comparison["volume_match"] = np.isclose(
    df_comparison["total_volume_m3_ifc"],
    df_comparison["total_volume_m3_revit"],
    atol=VOLUME_TOLERANCE,
    rtol=0
)

df_comparison["status"] = np.where(
    (df_comparison["_merge"] == "both")
    & df_comparison["count_match"]
    & df_comparison["volume_match"]
    & df_comparison["missing_volumes_ifc"].eq(0)
    & df_comparison["missing_volumes_revit"].eq(0),
    "PASSED",
    "FAILED"
)

display(
    df_comparison[[
        "category",
        "element_count_ifc",
        "element_count_revit",
        "total_volume_m3_ifc",
        "total_volume_m3_revit",
        "volume_difference_m3",
        "status"
    ]].round(3)
)

,category,element_count_ifc,element_count_revit,total_volume_m3_ifc,total_volume_m3_revit,volume_difference_m3,status
0,Beams,6,6,83.328,83.328,0.0,PASSED
1,Bridge Deck,1,1,3645.000,3645.000,0.0,PASSED
2,Columns,12,12,89.856,89.856,0.0,PASSED
3,Foundations,12,12,288.000,288.000,-0.0,PASSED
4,Walls,2,2,442.260,442.260,-0.0,PASSED


In [14]:
comparison_summary = {
    "categories_compared": len(df_comparison),

    "passed_categories": int(
        df_comparison["status"].eq("PASSED").sum()
    ),

    "failed_categories": int(
        df_comparison["status"].eq("FAILED").sum()
    ),

    "total_ifc_volume_m3": (
        df_ifc_qto["total_volume_m3"].sum()
    ),

    "total_revit_volume_m3": (
        df_revit_qto["total_volume_m3"].sum()
    ),

    "total_absolute_difference_m3": (
        df_comparison["volume_difference_m3"]
        .abs()
        .sum(min_count=len(df_comparison))
    ),

    "volume_tolerance_m3": VOLUME_TOLERANCE,

    "agreement_rate_pct": (
        df_comparison["status"].eq("PASSED").mean() * 100
    )
}

display(
    pd.DataFrame(
        comparison_summary.items(),
        columns=["metric", "value"]
    ).round(6)
)

,metric,value
0,categories_compared,5.000
1,passed_categories,5.000
2,failed_categories,0.000
3,total_ifc_volume_m3,4548.444
4,total_revit_volume_m3,4548.444
5,total_absolute_difference_m3,0.000
6,volume_tolerance_m3,0.001
7,agreement_rate_pct,100.000


In [15]:
df_revit.to_csv(
    PROCESSED_DIR / "revit_elements.csv",
    index=False
)

df_revit_qto.to_csv(
    PROCESSED_DIR / "revit_quantity_summary.csv",
    index=False
)

pd.DataFrame([csv_quality_summary]).to_csv(
    PROCESSED_DIR / "revit_quality_summary.csv",
    index=False
)

df_comparison.to_csv(
    PROCESSED_DIR / "ifc_revit_comparison.csv",
    index=False
)

pd.DataFrame([comparison_summary]).to_csv(
    PROCESSED_DIR / "ifc_revit_validation_summary.csv",
    index=False
)

print("All CSV and validation datasets exported successfully!")

All CSV and validation datasets exported successfully!
